# Capítulo 3 · Pandas desde cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_03/02_pandas.ipynb)

**Preparación de datos para Machine Learning**

---

### ¿Por qué Pandas?

Se suele decir que un científico de datos dedica **el 80% de su tiempo a preparar datos** y solo el 20% a entrenar modelos. Los datos del mundo real llegan:
- en archivos CSV, Excel, bases de datos…
- con **valores faltantes**, duplicados y errores de escritura,
- con texto, fechas y categorías que un modelo no entiende directamente.

**Pandas** es la herramienta estándar para **cargar, explorar, limpiar y transformar** datos tabulares. Está construida sobre NumPy y se integra con Matplotlib y scikit-learn.

> 🧩 Piensa en pandas como "**Excel con superpoderes y programable**": cada tabla es un `DataFrame`, cada columna es una `Series`.

### 🎯 Objetivos
1. Dominar las dos estructuras básicas: `Series` y `DataFrame`.
2. Cargar, explorar, filtrar y transformar datos.
3. Agrupar, combinar tablas y trabajar con fechas.

### 📚 Índice
1. `Series`
2. `DataFrame`
3. Cargar datos: nuestro dataset de viviendas
4. Explorar un dataset
5. Seleccionar y filtrar
6. Crear y modificar columnas
7. Agrupar y resumir (`groupby`, `pivot_table`)
8. Combinar tablas (`merge`, `concat`)
9. Fechas

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.5


> 💡 Convención universal: `import pandas as pd`.

---
## 1. `Series`: una columna con etiquetas

Una `Series` es un array de NumPy de 1 dimensión **con un índice** (etiquetas para cada valor).

In [5]:
temperaturas = pd.Series([22.5, 25.1, 19.8, 30.2],
                         index=["lunes", "martes", "miércoles", "jueves"],
                         name="temperatura")
temperaturas

lunes        22.5
martes       25.1
miércoles    19.8
jueves       30.2
Name: temperatura, dtype: float64

In [6]:
print(temperaturas["martes"])        # acceder por etiqueta
print(temperaturas.values)           # el array de NumPy que hay debajo
print(temperaturas.index)

25.1
[22.5 25.1 19.8 30.2]
Index(['lunes', 'martes', 'miércoles', 'jueves'], dtype='str')


In [7]:
# Las operaciones vectorizadas funcionan igual que en NumPy
print(temperaturas * 9 / 5 + 32)     # a Fahrenheit
print(temperaturas[temperaturas > 22])
print("Media:", temperaturas.mean())

lunes        72.50
martes       77.18
miércoles    67.64
jueves       86.36
Name: temperatura, dtype: float64
lunes     22.5
martes    25.1
jueves    30.2
Name: temperatura, dtype: float64
Media: 24.400000000000002


In [8]:
# value_counts: ¿cuántas veces aparece cada valor? (MUY usado con variables categóricas)
colores = pd.Series(["rojo", "azul", "rojo", "verde", "rojo", "azul"])
colores.value_counts()

rojo     3
azul     2
verde    1
Name: count, dtype: int64

---
## 2. `DataFrame`: una tabla

Un `DataFrame` es una tabla de 2 dimensiones: **filas** (con un índice) y **columnas** (cada una es una `Series` y puede tener un tipo distinto).

In [9]:
# Crear un DataFrame a partir de un diccionario: clave = nombre de columna
df = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Marta", "Pedro", "Lucía"],
    "edad": [28, 35, 22, 41, 30],
    "ciudad": ["Madrid", "Lima", "Bogotá", "Lima", "Madrid"],
    "salario": [32000, 45000, 28000, 52000, 39000],
})
df

,nombre,edad,ciudad,salario
0,Ana,28,Madrid,32000
1,Luis,35,Lima,45000
2,Marta,22,Bogotá,28000
3,Pedro,41,Lima,52000
4,Lucía,30,Madrid,39000


In [10]:
print("Forma:", df.shape)            # (filas, columnas)
print("Columnas:", list(df.columns))
print("Índice:", df.index)
print()
print(df.dtypes)                     # tipo de cada columna

Forma: (5, 4)
Columnas: ['nombre', 'edad', 'ciudad', 'salario']
Índice: RangeIndex(start=0, stop=5, step=1)

nombre       str
edad       int64
ciudad       str
salario    int64
dtype: object


In [11]:
# También desde un array de NumPy (lo típico al salir de un modelo)
matriz = np.random.default_rng(0).normal(size=(4, 3))
pd.DataFrame(matriz, columns=["feature_1", "feature_2", "feature_3"])

,feature_1,feature_2,feature_3
0,0.125730,-0.132105,0.640423
1,0.104900,-0.535669,0.361595
2,1.304000,0.947081,-0.703735
3,-1.265421,-0.623274,0.041326


In [12]:
# Y de vuelta a NumPy (lo que necesita un modelo de ML)
df[["edad", "salario"]].to_numpy()

array([[   28, 32000],
       [   35, 45000],
       [   22, 28000],
       [   41, 52000],
       [   30, 39000]])

---
## 3. Cargar datos: nuestro dataset de viviendas 🏠

Trabajaremos con un dataset de ventas de viviendas que está en `data/viviendas.csv`. Si todavía no tienes los datos del curso, descárgalos desde la raíz del repo:

```bash
uv run python scripts/download_data.py
```

In [13]:
# Los datos del curso están en la carpeta data/, en la raíz del repo.
# Para encontrar la raíz, subimos de carpeta en carpeta hasta dar con pyproject.toml.
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

Carpeta de datos: c:\Users\flavs\Documents\ml-course\data


Cargamos el CSV con `pd.read_csv`, como harías con cualquier dataset real:

In [14]:
df = pd.read_csv(DATA_DIR / "viviendas.csv")
df

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
0,1,2022-08-04,Este,127.0,3,6.0,sí,331800.0
1,2,2023-08-12,Norte,82.0,2,44.0,no,241300.0
2,3,2024-10-02,Oeste,101.0,4,13.0,no,265000.0
3,4,2023-06-18,Este,30.0,1,53.0,no,55500.0
4,5,2023-01-17,Centro,55.0,1,43.0,no,171500.0
...,...,...,...,...,...,...,...,...
595,596,2024-11-30,Norte,86.0,1,13.0,sí,239900.0
596,597,2022-03-22,Este,60.0,2,19.0,no,126200.0
597,598,2022-09-28,Sur,87.0,1,26.0,no,181700.0
598,599,2022-11-26,Oeste,42.0,1,49.0,no,97400.0


> 📝 **Otras formas de cargar datos** (para que las conozcas):
> ```python
> pd.read_csv("https://url/a/un/archivo.csv")      # directamente desde internet
> pd.read_csv("archivo.csv", sep=";", decimal=",")  # CSV con formato europeo/latino
> pd.read_excel("archivo.xlsx", sheet_name="Hoja1") # Excel (requiere openpyxl)
> pd.read_json("archivo.json")
> pd.read_parquet("archivo.parquet")                # formato muy eficiente para big data
> pd.read_sql("SELECT * FROM tabla", conexion)      # bases de datos
> ```

---
## 4. Explorar un dataset

Lo **primero** que se hace con cualquier dataset nuevo. Estas funciones son tu "radiografía" de los datos:

In [15]:
df.head()          # primeras 5 filas (df.head(10) para 10)

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
0,1,2022-08-04,Este,127.0,3,6.0,sí,331800.0
1,2,2023-08-12,Norte,82.0,2,44.0,no,241300.0
2,3,2024-10-02,Oeste,101.0,4,13.0,no,265000.0
3,4,2023-06-18,Este,30.0,1,53.0,no,55500.0
4,5,2023-01-17,Centro,55.0,1,43.0,no,171500.0


In [16]:
df.tail(3)         # últimas filas

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
597,598,2022-09-28,Sur,87.0,1,26.0,no,181700.0
598,599,2022-11-26,Oeste,42.0,1,49.0,no,97400.0
599,600,2022-10-01,Sur,67.0,2,44.0,no,143600.0


In [17]:
df.sample(5, random_state=0)    # filas aleatorias (a veces el inicio no es representativo)

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
434,435,2024-12-23,Centro,133.0,3,24.0,no,482000.0
122,123,2022-10-17,Centro,84.0,2,59.0,sí,289900.0
224,225,2023-04-21,Norte,57.0,2,20.0,sí,140000.0
479,480,2024-06-10,Norte,94.0,4,53.0,no,272000.0
205,206,2024-07-05,Norte,144.0,5,12.0,no,422500.0


In [18]:
df.info()          # columnas, tipos, y ¡cuántos valores NO nulos hay!

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            600 non-null    int64  
 1   fecha_venta   600 non-null    str    
 2   barrio        600 non-null    str    
 3   metros        600 non-null    float64
 4   habitaciones  600 non-null    int64  
 5   antiguedad    600 non-null    float64
 6   garaje        600 non-null    str    
 7   precio        600 non-null    float64
dtypes: float64(3), int64(2), str(3)
memory usage: 47.6 KB


> 🔍 Fíjate en `info()`: hay **600 filas** y ninguna columna tiene valores nulos. Pero `fecha_venta` aparece como texto (`str`), no como fecha: la convertiremos en la sección de fechas.

In [19]:
df.describe()      # estadísticas de las columnas numéricas

,id,metros,habitaciones,antiguedad,precio
count,600.000000,600.000000,600.000000,600.000000,600.000000
mean,300.500000,89.330000,2.591667,29.310000,247229.666667
std,173.349358,29.265313,1.216853,17.218828,99236.668191
min,1.000000,30.000000,1.000000,0.000000,19700.000000
25%,150.750000,69.750000,2.000000,14.000000,171950.000000
50%,300.500000,90.000000,3.000000,28.000000,241200.000000
75%,450.250000,108.000000,3.000000,45.000000,309150.000000
max,600.000000,185.000000,6.000000,59.000000,647900.000000


> 🔍 Con `describe()` ves de un vistazo la escala de cada variable: cuántos valores hay (`count`), la media, la dispersión (`std`) y el rango (`min`, `max`).

In [20]:
df.describe(exclude="number")     # resumen de las columnas no numéricas

,fecha_venta,barrio,garaje
count,600,600,600
unique,466,5,2
top,2022-11-09,Centro,no
freq,4,134,350


In [21]:
df["barrio"].value_counts()

barrio
Centro    134
Norte     133
Sur       126
Oeste     114
Este       93
Name: count, dtype: int64

In [22]:
df.nunique()       # número de valores distintos por columna

id              600
fecha_venta     466
barrio            5
metros          123
habitaciones      6
antiguedad       60
garaje            2
precio          556
dtype: int64

---
## 5. Seleccionar y filtrar

### 5.1 Seleccionar columnas

In [23]:
df["precio"].head()                    # una columna → Series

0    331800.0
1    241300.0
2    265000.0
3     55500.0
4    171500.0
Name: precio, dtype: float64

In [24]:
df[["barrio", "metros", "precio"]].head()   # varias columnas → DataFrame (¡doble corchete!)

,barrio,metros,precio
0,Este,127.0,331800.0
1,Norte,82.0,241300.0
2,Oeste,101.0,265000.0
3,Este,30.0,55500.0
4,Centro,55.0,171500.0


### 5.2 Seleccionar filas: `loc` e `iloc` ⭐

| | `loc` | `iloc` |
|---|---|---|
| Selecciona por… | **etiquetas** (nombres del índice y de columnas) | **posiciones** (números enteros, como en NumPy) |
| Ejemplo | `df.loc[5, "precio"]` | `df.iloc[5, 7]` |
| Slicing | incluye el final | excluye el final |

In [25]:
df.iloc[0]                 # primera fila (por posición)

id                       1
fecha_venta     2022-08-04
barrio                Este
metros               127.0
habitaciones             3
antiguedad             6.0
garaje                  sí
precio            331800.0
Name: 0, dtype: object

In [26]:
df.iloc[:3, :4]            # 3 primeras filas, 4 primeras columnas

,id,fecha_venta,barrio,metros
0,1,2022-08-04,Este,127.0
1,2,2023-08-12,Norte,82.0
2,3,2024-10-02,Oeste,101.0


In [27]:
df.loc[:4, ["barrio", "precio"]]     # filas con índice 0 a 4 (¡incluido!) y columnas por nombre

,barrio,precio
0,Este,331800.0
1,Norte,241300.0
2,Oeste,265000.0
3,Este,55500.0
4,Centro,171500.0


### 5.3 Filtrar con condiciones (máscaras booleanas) ⭐

Exactamente igual que en NumPy:

In [28]:
df[df["precio"] > 400_000].head()

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
8,9,2024-12-05,Centro,150.0,5,5.0,no,554700.0
15,16,2022-05-16,Norte,145.0,3,17.0,no,408200.0
25,26,2024-03-05,Centro,128.0,5,10.0,no,492500.0
50,51,2023-03-28,Centro,131.0,4,6.0,no,459400.0
54,55,2023-02-21,Este,166.0,5,26.0,sí,422400.0


In [29]:
# Varias condiciones: & (y), | (o), ~ (no). ¡Paréntesis obligatorios!
grandes_con_garaje = df[(df["metros"] > 150) & (df["garaje"] == "sí")]
print(len(grandes_con_garaje), "viviendas grandes con garaje")
grandes_con_garaje.head()

5 viviendas grandes con garaje


,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
54,55,2023-02-21,Este,166.0,5,26.0,sí,422400.0
176,177,2023-11-18,Norte,151.0,6,46.0,sí,475300.0
380,381,2024-02-12,Sur,157.0,4,21.0,sí,346200.0
476,477,2022-07-14,Norte,163.0,6,18.0,sí,497400.0
500,501,2023-09-12,Este,174.0,5,7.0,sí,438200.0


In [30]:
# isin: ¿el valor está en una lista?
df[df["barrio"].isin(["Norte", "Sur"])].head()

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
1,2,2023-08-12,Norte,82.0,2,44.0,no,241300.0
9,10,2023-08-01,Sur,55.0,1,57.0,no,77100.0
10,11,2022-05-31,Norte,65.0,2,48.0,no,162300.0
12,13,2024-07-20,Sur,138.0,5,16.0,sí,285200.0
14,15,2024-04-15,Norte,97.0,3,23.0,no,279200.0


In [31]:
# between: rango de valores (incluye los extremos)
df[df["antiguedad"].between(0, 5)].shape

(51, 8)

In [32]:
# query: una alternativa más legible para filtros complejos
df.query("habitaciones >= 4 and precio < 300000").head()

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
2,3,2024-10-02,Oeste,101.0,4,13.0,no,265000.0
11,12,2022-03-15,Oeste,97.0,4,9.0,no,263400.0
12,13,2024-07-20,Sur,138.0,5,16.0,sí,285200.0
30,31,2024-07-15,Este,103.0,4,48.0,no,244700.0
44,45,2022-07-28,Oeste,91.0,4,25.0,sí,272900.0


In [33]:
# Filtrar Y seleccionar columnas a la vez con loc
df.loc[df["barrio"] == "Centro", ["metros", "precio"]].head()

,metros,precio
4,55.0,171500.0
8,150.0,554700.0
17,52.0,174300.0
25,128.0,492500.0
27,71.0,235500.0


### 5.4 Ordenar

In [34]:
df.sort_values("precio", ascending=False).head()

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
269,270,2022-07-09,Centro,185.0,6,47.0,no,647900.0
589,590,2022-09-09,Centro,164.0,5,26.0,no,605300.0
8,9,2024-12-05,Centro,150.0,5,5.0,no,554700.0
265,266,2023-02-15,Centro,143.0,4,27.0,sí,545300.0
263,264,2023-08-01,Centro,147.0,5,11.0,no,540500.0


In [35]:
df.sort_values(["barrio", "precio"], ascending=[True, False]).head()

,id,fecha_venta,barrio,metros,habitaciones,antiguedad,garaje,precio
269,270,2022-07-09,Centro,185.0,6,47.0,no,647900.0
589,590,2022-09-09,Centro,164.0,5,26.0,no,605300.0
8,9,2024-12-05,Centro,150.0,5,5.0,no,554700.0
265,266,2023-02-15,Centro,143.0,4,27.0,sí,545300.0
263,264,2023-08-01,Centro,147.0,5,11.0,no,540500.0


---
## 6. Crear y modificar columnas

In [36]:
# Para no estropear los datos originales, trabajamos sobre una copia
ejemplo = df.head(8).copy()

# Nueva columna a partir de otras (operación vectorizada)
ejemplo["precio_m2"] = ejemplo["precio"] / ejemplo["metros"]

# Columna condicional con np.where
ejemplo["es_nueva"] = np.where(ejemplo["antiguedad"] < 10, "nueva", "usada")

ejemplo[["metros", "precio", "precio_m2", "antiguedad", "es_nueva"]]

,metros,precio,precio_m2,antiguedad,es_nueva
0,127.0,331800.0,2612.598425,6.0,nueva
1,82.0,241300.0,2942.682927,44.0,usada
2,101.0,265000.0,2623.762376,13.0,usada
3,30.0,55500.0,1850.000000,53.0,usada
4,55.0,171500.0,3118.181818,43.0,usada
5,81.0,218200.0,2693.827160,16.0,usada
6,58.0,116700.0,2012.068966,33.0,usada
7,111.0,271300.0,2444.144144,6.0,nueva


In [37]:
# map: sustituir valores usando un diccionario (ideal para codificar categorías)
ejemplo["garaje_num"] = ejemplo["garaje"].map({"sí": 1, "no": 0})

# apply: aplicar una función propia a cada valor
def tamaño(m):
    if m < 60:
        return "pequeña"
    elif m < 120:
        return "mediana"
    return "grande"

ejemplo["tamaño"] = ejemplo["metros"].apply(tamaño)
ejemplo[["garaje", "garaje_num", "metros", "tamaño"]]

,garaje,garaje_num,metros,tamaño
0,sí,1,127.0,grande
1,no,0,82.0,mediana
2,no,0,101.0,mediana
3,no,0,30.0,pequeña
4,no,0,55.0,pequeña
5,no,0,81.0,mediana
6,no,0,58.0,pequeña
7,no,0,111.0,mediana


> ⚡ **Rendimiento**: `apply` es en realidad un bucle de Python, así que es lento con millones de filas. Siempre que puedas, usa operaciones vectorizadas (`+`, `*`, `np.where`, `.str`, `.dt`, `pd.cut`…).

In [38]:
# pd.cut: convertir una variable numérica en categorías por rangos (binning) — ¡vectorizado!
ejemplo["tamaño_v2"] = pd.cut(ejemplo["metros"], bins=[0, 60, 120, np.inf],
                              labels=["pequeña", "mediana", "grande"])
ejemplo[["metros", "tamaño", "tamaño_v2"]]

,metros,tamaño,tamaño_v2
0,127.0,grande,grande
1,82.0,mediana,mediana
2,101.0,mediana,mediana
3,30.0,pequeña,pequeña
4,55.0,pequeña,pequeña
5,81.0,mediana,mediana
6,58.0,pequeña,pequeña
7,111.0,mediana,mediana


In [39]:
# Renombrar y eliminar columnas
ejemplo = ejemplo.rename(columns={"metros": "superficie_m2"})
ejemplo = ejemplo.drop(columns=["tamaño_v2", "es_nueva"])
ejemplo.columns

Index(['id', 'fecha_venta', 'barrio', 'superficie_m2', 'habitaciones',
       'antiguedad', 'garaje', 'precio', 'precio_m2', 'garaje_num', 'tamaño'],
      dtype='str')

> ⚠️ **Buena práctica**: la mayoría de métodos de pandas **devuelven un DataFrame nuevo** y no modifican el original. Por eso escribimos `df = df.metodo(...)`. Para modificar valores concretos usa siempre `df.loc[filas, columna] = valor`.

---
## 7. Agrupar y resumir ⭐

`groupby` sigue el patrón **dividir → aplicar → combinar**: separa las filas en grupos, calcula algo en cada grupo y junta los resultados.

In [40]:
df.groupby("barrio")["precio"].mean().sort_values(ascending=False)

barrio
Centro    326817.910448
Norte     248521.052632
Oeste     242145.614035
Este      221379.569892
Sur       184904.761905
Name: precio, dtype: float64

In [41]:
# Varias estadísticas a la vez con agg
df.groupby("barrio")["precio"].agg(["count", "mean", "median", "min", "max"]).round(0)

,count,mean,median,min,max
barrio,,,,,
Centro,134,326818.0,333600.0,105800.0,647900.0
Este,93,221380.0,226000.0,55500.0,438200.0
Norte,133,248521.0,249300.0,62000.0,497400.0
Oeste,114,242146.0,243350.0,86500.0,423300.0
Sur,126,184905.0,184250.0,19700.0,346200.0


In [42]:
# Estadísticas distintas para columnas distintas, con nombres personalizados
resumen = df.groupby("barrio").agg(
    n_ventas=("id", "count"),
    precio_medio=("precio", "mean"),
    metros_medios=("metros", "mean"),
    pct_garaje=("garaje", lambda s: (s == "sí").mean() * 100),
).round(1)
resumen

,n_ventas,precio_medio,metros_medios,pct_garaje
barrio,,,,
Centro,134,326817.9,91.2,41.8
Este,93,221379.6,88.7,37.6
Norte,133,248521.1,86.8,45.9
Oeste,114,242145.6,90.2,44.7
Sur,126,184904.8,89.6,37.3


In [43]:
# Agrupar por varias columnas
df.groupby(["barrio", "garaje"])["precio"].mean().round(0).unstack()

garaje,no,sí
barrio,,
Centro,316994.0,340502.0
Este,213993.0,233620.0
Norte,241911.0,256323.0
Oeste,234451.0,251651.0
Sur,171977.0,206634.0


### Tablas dinámicas y tablas cruzadas

In [44]:
# pivot_table: como las tablas dinámicas de Excel
pd.pivot_table(df, values="precio", index="barrio", columns="habitaciones",
               aggfunc="mean").round(-3)

habitaciones,1,2,3,4,5,6
barrio,,,,,,
Centro,206000.0,313000.0,345000.0,400000.0,505000.0,648000.0
Este,137000.0,187000.0,244000.0,266000.0,367000.0,NaN
Norte,173000.0,214000.0,284000.0,315000.0,401000.0,486000.0
Oeste,158000.0,217000.0,260000.0,301000.0,368000.0,NaN
Sur,107000.0,165000.0,204000.0,257000.0,268000.0,345000.0


In [45]:
# crosstab: frecuencias cruzadas entre dos variables categóricas
pd.crosstab(df["barrio"], df["garaje"], normalize="index").round(2)   # proporciones por fila

garaje,no,sí
barrio,,
Centro,0.58,0.42
Este,0.62,0.38
Norte,0.54,0.46
Oeste,0.55,0.45
Sur,0.63,0.37


---
## 8. Combinar tablas

Los datos suelen estar repartidos en varias tablas. Imagina que tenemos otra tabla con información de cada barrio:

In [46]:
info_barrios = pd.DataFrame({
    "barrio": ["Centro", "Norte", "Sur", "Este", "Oeste", "Puerto"],
    "distancia_centro_km": [0.0, 5.5, 7.0, 4.0, 6.0, 12.0],
    "indice_seguridad": [7.2, 8.5, 6.1, 7.8, 8.0, 6.5],
})
info_barrios

,barrio,distancia_centro_km,indice_seguridad
0,Centro,0.0,7.2
1,Norte,5.5,8.5
2,Sur,7.0,6.1
3,Este,4.0,7.8
4,Oeste,6.0,8.0
5,Puerto,12.0,6.5


### `merge`: unir por una columna en común (como un `JOIN` de SQL)

| `how=` | Resultado |
|---|---|
| `"inner"` | solo las filas cuya clave existe en **ambas** tablas |
| `"left"` | todas las filas de la tabla izquierda (lo más habitual) |
| `"right"` | todas las de la derecha |
| `"outer"` | todas las filas de ambas |

In [47]:
df = df.merge(info_barrios, on="barrio", how="left")
df[["barrio", "precio", "distancia_centro_km", "indice_seguridad"]].head()

,barrio,precio,distancia_centro_km,indice_seguridad
0,Este,331800.0,4.0,7.8
1,Norte,241300.0,5.5,8.5
2,Oeste,265000.0,6.0,8.0
3,Este,55500.0,4.0,7.8
4,Centro,171500.0,0.0,7.2


> 💡 "Puerto" no aparece en nuestras ventas, así que con `how="left"` simplemente se ignora.

### `concat`: apilar tablas

In [48]:
ventas_enero = pd.DataFrame({"producto": ["A", "B"], "unidades": [10, 5]})
ventas_febrero = pd.DataFrame({"producto": ["A", "C"], "unidades": [7, 12]})

pd.concat([ventas_enero, ventas_febrero], ignore_index=True)

,producto,unidades
0,A,10
1,B,5
2,A,7
3,C,12


---
## 9. Fechas

Al cargar el CSV, `fecha_venta` llegó como texto. Primero la convertimos a fecha con `pd.to_datetime`:

In [49]:
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"])
df.dtypes

id                              int64
fecha_venta            datetime64[us]
barrio                            str
metros                        float64
habitaciones                    int64
antiguedad                    float64
garaje                            str
precio                        float64
distancia_centro_km           float64
indice_seguridad              float64
dtype: object

Con la columna convertida a `datetime`, el accesor `.dt` nos da acceso a sus componentes:

In [50]:
df["año"] = df["fecha_venta"].dt.year
df["mes"] = df["fecha_venta"].dt.month
df["dia_semana"] = df["fecha_venta"].dt.dayofweek      # 0 = lunes, 6 = domingo
df[["fecha_venta", "año", "mes", "dia_semana"]].head()

,fecha_venta,año,mes,dia_semana
0,2022-08-04,2022,8,3
1,2023-08-12,2023,8,5
2,2024-10-02,2024,10,2
3,2023-06-18,2023,6,6
4,2023-01-17,2023,1,1


In [51]:
# Filtrar por fechas
df[df["fecha_venta"] >= "2024-06-01"].shape

(130, 13)

In [52]:
# Serie temporal: precio medio por mes ("ME" = fin de mes)
precio_mensual = df.set_index("fecha_venta")["precio"].resample("ME").mean()
precio_mensual.head()

fecha_venta
2022-01-31    248392.857143
2022-02-28    226280.000000
2022-03-31    234462.500000
2022-04-30    239416.666667
2022-05-31    223290.000000
Freq: ME, Name: precio, dtype: float64